In [2]:
import torch
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

torch.manual_seed(42)

r = torch.rand(size = (1000,1)) * 5

b = torch.pi * torch.square(r)
pow = 3
powers = torch.arange(start = 0, end = pow, step = 1)
B = torch.pow(r,powers)

def FISTA(Vandermonde, y, epochs = 1000, lamb = 0.5, grad = lambda A, x, b: (A.T @ (A @ x - b)), cutoff = 0.5, lr = None): ## lambda is arbitrarily set but you can change it. cutoff is the cutoff value for doing the least squares at the end.
  x = torch.rand(size = (Vandermonde.shape[1],)).reshape(-1,1)
  if lr == None:
    lr = 1 / (torch.linalg.norm(Vandermonde.T @ Vandermonde, ord = 2)) ## lr guareenteeing convergence for fista

  Vandermonde = Vandermonde
  soft_thres = lambda x, th = lamb * lr: torch.sign(x) * torch.maximum(torch.abs(x) - th, torch.tensor(0))
  prev_step = torch.zeros(Vandermonde.shape[1]).reshape(-1,1)

  tk = 1
  losses = []
  epochs_list = []
  for epoch in range(epochs):
    Beta = (tk - 1) / ((1 + sqrt(1 + 4 * tk ** 2)) / 2)
    prev_step = Beta * prev_step - lr * grad(Vandermonde, x + prev_step, y)
    x += prev_step
    x = soft_thres(x)

    tk = (1 + sqrt(1 + 4 * tk ** 2)) / 2

    losses.append((torch.linalg.vector_norm((A @ x - b), ord = 2).square() / (A @ x - b).size(0)).item()) ## MSE loss
    epochs_list.append(epoch)

  return epochs_list, losses, x

  ## We are comparing without the refining adjustment stage
  active = np.abs(x) > cutoff

  x_debiased = np.zeros_like(x)
  x_debiased[active] = np.linalg.lstsq(
      Vandermonde[:, active.reshape(-1)],
      y,
      rcond=None
  )[0].reshape(-1)

  return x_debiased


A = torch.concat([B,torch.sin(r), torch.cos(r), torch.special.erf(r), torch.special.airy_ai(r), torch.special.bessel_j0(r), torch.special.ndtr(r), torch.special.legendre_polynomial_p(r,3)], dim = 1)
FISTA_CIR_epochs, FISTA_CIR_losses, FISTA_CIR_x = FISTA(A, b, epochs= 100000, lamb = 3e-1)


r = torch.rand(size = (100000,1)) * 5
h = torch.rand(size = (100000,1)) * 5
b = torch.pi * torch.square(r) * h

pow = 3
powers = torch.arange(start = 0, end = pow, step = 1)

B = (torch.pow(r,torch.meshgrid(powers, powers)[0].reshape(-1,)) * torch.pow(h,torch.meshgrid(powers, powers)[1].reshape(-1,)))

A = torch.concat([B,torch.sin(r), torch.cos(r), torch.special.erf(r), torch.special.airy_ai(r), torch.special.bessel_j0(r), torch.special.ndtr(r), torch.special.legendre_polynomial_p(r,3)], dim = 1)
FISTA_CYL_epochs, FISTA_CYL_losses, FISTA_CYL_x = FISTA(A, b, epochs= 100000, lamb = 3e-1)

c = (5) * torch.rand(size = (100000,1)) - 2.5
d = (5) * torch.rand(size = (100000,1)) - 2.5

b = torch.sin(c ** 2 + d ** 2)

pow = 3
powers = torch.arange(start = 0, end = pow, step = 1)

B = (torch.pow(c,torch.meshgrid(powers, powers)[0].reshape(-1,)) * torch.pow(d,torch.meshgrid(powers, powers)[1].reshape(-1,)))

import itertools

## Getting all possible polynomials of Q_3,3

exps = [(i, j) for i in range(3) for j in range(3)]

expressions = []

for i, j in exps:
    m = (c ** i) * (d ** j)

    expressions.append(torch.sin(m))

for (i1, j1), (i2, j2) in itertools.combinations(exps, 2):
    m1 = (c ** i1) * (d ** j1)

    m2 = (c ** i2) * (d ** j2)
    expressions.append(torch.sin(m1 + m2))

for (i1, j1), (i2, j2), (i3, j3) in itertools.combinations(exps, 3):
    m1 = (c ** i1) * (d ** j1)

    m2 = (c ** i2) * (d ** j2)

    m3 = (c ** i3) * (d ** j3)
    expressions.append(torch.sin(m1 + m2 + m3))

## equivalent formula for SSR
## We remove the first column of the torch stack because it is of constant sin(1), which is linearly dependent to B's first column of ones
A = torch.concat([B,torch.stack(expressions).squeeze()[1:,:].T], dim = 1)
FISTA_SIN_epochs, FISTA_SIN_losses, FISTA_SIN_x = FISTA(A, b, epochs= 100000, lamb = 3e-1)



import torch
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

%matplotlib inline


fig, axes = plt.subplots(1,3, figsize = (24,6))

expected = np.zeros_like(FISTA_CIR_x)
expected[2] = np.pi
axes[0].bar(x=np.arange(len(FISTA_CIR_x)), height=FISTA_CIR_x.squeeze().tolist(), color = [{False: "indianred", True:"teal"}[j.item() != 0] for j in expected])
axes[0].axhline(y = 0, color = "black", linewidth = 2)
for i, exp in enumerate(expected):
    y = 0
    if (exp.item() - FISTA_CIR_x[i] < 0).item():
        y = FISTA_CIR_x[i]
    else:
        y = exp.item()

    axes[0].hlines(y=exp,xmin=i - 0.4,xmax=i + 0.4,color='orange',linewidth=2)
    if (exp.item() - FISTA_CIR_x[i]).abs() >= 0.1:
        axes[0].text(i - 0.4,y + 0.03,f"{(exp.item() - FISTA_CIR_x[i]).abs().item():.3f}")

axes[0].set_xlabel("Coefficient Number")
axes[0].set_ylabel("Regressed Coefficients")
axes[0].set_title("Area of Circle SSR Coefficients")
axes[0].set_xticks(np.arange(len(FISTA_CIR_x)))
axes[0].tick_params(axis='x', labelsize=10)

expected = np.zeros_like(FISTA_CYL_x)
expected[7] = np.pi
axes[1].bar(x=np.arange(len(FISTA_CYL_x)), height=FISTA_CYL_x.squeeze().tolist(), color = [{False: "indianred", True:"teal"}[j.item() != 0] for j in expected])
axes[1].axhline(y = 0, color = "black", linewidth = 2)
for i, exp in enumerate(expected):
    y = 0
    if exp.item() - FISTA_CYL_x[i] < 0:
        y = FISTA_CYL_x[i]
    else:
        y = exp.item()


    axes[1].hlines(y=exp,xmin=i - 0.4,xmax=i + 0.4,color='orange',linewidth=2)
    if (exp.item() - FISTA_CYL_x[i]).abs() >= 0.1:
        axes[1].text(i - 0.5,y + 0.03,f"{(exp.item() - FISTA_CYL_x[i]).abs().item():.3f}", fontsize = 8)

axes[1].set_xlabel("Coefficient Number")
axes[1].set_ylabel("Regressed Coefficients")
axes[1].set_title("Volume of Cylinder SSR Coefficients")
axes[1].set_xticks(np.arange(len(FISTA_CYL_x)))
axes[1].tick_params(axis='x', labelsize=10)


expected = np.zeros_like(FISTA_SIN_x)
expected[-102] = 1
axes[2].bar(x=np.arange(len(FISTA_SIN_x)), height=FISTA_SIN_x.squeeze().tolist(), color = [{False: "indianred", True:"teal"}[j.item() != 0] for j in expected])
axes[2].axhline(y = 0, color = "black", linewidth = 2)
for i, exp in enumerate(expected):
    axes[2].hlines(y=exp,xmin=i - 0.4,xmax=i + 0.4,color='orange',linewidth=2)

    if (exp.item() - FISTA_SIN_x[i]).abs() >= 0.1:
        y = 0
        if exp.item() - FISTA_SIN_x[i] < 0:
            y = FISTA_SIN_x[i]
        else:
            y = exp.item()
        axes[2].text(i - 6,y + 0.01,f"{(exp.item() - FISTA_SIN_x[i]).abs().item():.3f}")

axes[2].set_xlabel("Coefficient Number")
axes[2].set_ylabel("Regressed Coefficients")
axes[2].set_title("$\\sin(x^2+y^2)$ SSR Coefficients")

axes[2].set_xticks(np.arange(start = 0, stop= len(FISTA_SIN_x), step = 10))
axes[2].tick_params(axis='x', labelsize=8)
axes[2].set_ylim((-0.3,1.1));

KeyboardInterrupt: 